# CamouBlend pilot M0/M1 trên Kaggle

**Chuẩn bị một lần**
1. Local: `bash scripts/pack_kaggle.sh` → upload `dist/camo-code.zip` thành dataset `camo-code` (hoặc New Version).
2. Notebook: *Settings → Accelerator GPU T4/P100, Internet On*. *Add Input*:
   - `camo-code`
   - `aisegmentcom-matting-human-datasets` (chân dung có alpha matte; tìm tên này trong Add Input)
   - (tuỳ chọn) DTD có sẵn trên Kaggle; không có thì code tự tải từ Oxford (~625 MB).
3. Tuỳ chọn: secret `HF_TOKEN`.

Input được lấy từ dataset công khai: cat/lizard/fish ← Open Images (mask), butterfly ← Leeds Butterfly (mask), portrait ← AISegment (random), nền ← DTD. 4 ứng viên mỗi ID, mặc định dùng ứng viên 0.

**Chạy 2 lần**
1. `STEP = "fetch-inputs"` → xem `candidates.png` ở cell dưới; muốn đổi ứng viên thì sửa `PICK`.
2. `STEP = "all"` (*Save Version → Save & Run All* cũng được) → chuẩn hoá → QA → 21 audit → 15 default → `pilot.zip`.

Bị ngắt: chạy lại, trial đã `ok` được bỏ qua. Đổi ứng viên sau khi đã render: xoá `/kaggle/working/pilot` rồi chạy lại.

In [ ]:
STEP = "fetch-inputs"   # fetch-inputs | all | preview | audit | default
PICK = {}               # đổi ứng viên, vd: {"O05": 2, "O04": 1, "B03": 3}
RUN = "pilot"           # tên thư mục/zip output; vd "pilot_v2" để không trộn với lần trước
CONFIGS = ""            # "" = bảng C00-C06 gốc | "pilot/audit_v2.yaml"
EXTRA = ""              # vd: "--pairs P02,P13"

import glob, os, re
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
CODE = next((os.path.dirname(p) for depth in range(1, 7) for p in sorted(glob.glob("/kaggle/input" + "/*" * depth + "/run.sh"))
             if os.path.isfile(os.path.join(os.path.dirname(p), "camo", "sources.py"))), None)  # no ** glob: AISegment is huge
assert CODE, "Không thấy code: attach dataset camo-code bản mới (có camo/sources.py)"
GEN, OUT = "/kaggle/working/pilot_inputs", f"/kaggle/working/{RUN}"
if CONFIGS:
    EXTRA += f" --configs {CODE}/{CONFIGS}"
INPUTS = f"{GEN}/inputs.yaml"
FROZEN = f"{CODE}/pilot/frozen_inputs"   # picked inputs shipped with the code: no dataset downloads
if STEP != "fetch-inputs" and not os.path.exists(INPUTS) and os.path.exists(f"{FROZEN}/inputs.yaml"):
    !mkdir -p {GEN} && cp -r {FROZEN}/. {GEN}/
if STEP == "fetch-inputs" or not os.path.exists(INPUTS):
    !cd {CODE} && bash run.sh pilot fetch-inputs --save {GEN}
if PICK:  # rewrite the chosen candidate paths
    text = open(INPUTS, encoding="utf-8").read()
    for item, k in PICK.items():
        text = re.sub(rf"candidates/{item}_\d+\.png", f"candidates/{item}_{k}.png", text)
    open(INPUTS, "w", encoding="utf-8").write(text)
print("code  :", CODE); print("inputs:", INPUTS)
if STEP != "fetch-inputs":
    steps = ["normalize", STEP] if STEP in ("preview", "audit", "default") else [STEP]
    for s in steps:
        for attempt in range(3):  # the watchdog exits on a hung render; reruns resume and skip finished trials
            !cd {CODE} && bash run.sh pilot {s} --inputs {INPUTS} --out {OUT} {EXTRA}
            if _exit_code == 0 or s in ("normalize", "preview"):  # retry only after a watchdog exit / failure
                break
    !rm -rf {OUT}/pilot_inputs && mkdir -p {OUT}/pilot_inputs && cp -r {GEN}/candidates {GEN}/inputs.yaml {GEN}/candidates.png {OUT}/pilot_inputs/
    !cd /kaggle/working && rm -f {RUN}.zip && zip -qr {RUN}.zip {RUN} && ls -lh {RUN}.zip

In [ ]:
# Xem nhanh
from IPython.display import Image as Show, display
for p in [f"{GEN}/candidates.png", f"{OUT}/inputs_overview.png"] + sorted(glob.glob(f"{OUT}/qa/*.png"))[:3] + sorted(glob.glob(f"{OUT}/runs/*/contact_sheet.png")):
    if os.path.exists(p):
        print(p); display(Show(filename=p, width=1000))